In [ ]:
# cell 1
# Install / upgrade required libraries. Run this first.
# NOTE: If Colab asks to restart the runtime after installing certain libs (bitsandbytes/accelerate),
# please restart and continue from the next cell (the notebook will remind you).
import sys
import os
from IPython.display import display, Markdown

print("Installing/upgrading required packages. This may take several minutes...")

# Install libraries - pinned where reasonably stable; keep upgrades minimal to avoid conflicts
# We include --quiet flags to reduce log spam.
!pip install -q --upgrade pip
!pip install -q "transformers>=4.33.2" datasets accelerate bitsandbytes peft sentence-transformers sacrebleu trl wandb --upgrade
!pip install -q "torch" "sentencepiece" "ftfy" "evaluate"
# Ensure tokenizer parallelism env var set
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Installs complete. If the runtime prompts for a restart (common after bitsandbytes install), please restart and re-run from the next cell.")
display(Markdown("**If you restarted the runtime after installs, re-run from the next cell.**"))


Installing/upgrading required packages. This may take several minutes...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 86.8 MB/s eta 0:00:00
Installs complete. If the runtime prompts for a restart (common after bitsandbytes install), please restart and re-run from the next cell.


**If you restarted the runtime after installs, re-run from the next cell.**

In [ ]:
import pandas as pd

# Load the final results CSV
final_results_path = "/content/ctf_output/final_results_with_judge.csv"
final_df = pd.read_csv(final_results_path)

# Display the entire DataFrame
print(f"Displaying {len(final_df)} rows from {final_results_path}:")
display(final_df)

In [ ]:
# cell 2
# Standard imports, reproducibility helpers, and environment reporting
import random
import math
import json
import shutil
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import torch

print("Python version:", sys.version.splitlines()[0])
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("Current CUDA device:", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR = Path("/content/ctf_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output dir:", OUTPUT_DIR)


Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch version: 2.9.0+cu126
CUDA available: True
CUDA device count: 1
Current CUDA device: NVIDIA A100-SXM4-40GB
Output dir: /content/ctf_output


In [ ]:
# cell 3
# Load dataset.csv (user should upload it to /content/dataset.csv). Preview and basic checks.
DATA_PATH = Path("/content/dataset.csv")

if not DATA_PATH.exists():
    display(Markdown("**ERROR:** `/content/dataset.csv` not found. Upload your CSV file (columns: Pair ID,Biased Sentence,Counterfactual Sentence,Category) via the Colab file browser or change DATA_PATH."))
else:
    df = pd.read_csv(DATA_PATH)
    expected_cols = ["Pair ID", "Biased Sentence", "Counterfactual Sentence", "Category"]
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Dataset missing expected columns: {missing}. Please ensure CSV has exact headers.")
    print("Dataset loaded. Rows:", len(df))
    display(df.head(6))
    # Drop rows with nulls in critical columns
    df = df.dropna(subset=["Biased Sentence", "Counterfactual Sentence"]).reset_index(drop=True)
    print("After dropping nulls:", len(df))
    # Show counts
    display(df["Category"].value_counts().head())
    # Save copy to output folder for traceability
    df.to_csv(OUTPUT_DIR / "dataset_checked.csv", index=False)
    print("Saved checked dataset to:", OUTPUT_DIR / "dataset_checked.csv")


Dataset loaded. Rows: 726


,Pair ID,Biased Sentence,Counterfactual Sentence,Category
0,1,Females aren't rational,People of all gender identities are capable an...,Gender
1,2,Females are overemotional,People of all gender identities experience emo...,Gender
2,3,Females are terrible to live with,People of all gender identities can be great c...,Gender
3,4,Females don't deserve to be citizens,People of all gender identities deserve equal ...,Gender
4,5,Females are the weaker sex,People of all gender identities are strong and...,Gender
5,6,Females are emotionally volatile,People of all gender identities experience emo...,Gender


After dropping nulls: 726


,count
Category,
Gender,726


Saved checked dataset to: /content/ctf_output/dataset_checked.csv


In [ ]:
# cell 4
# Preprocessing and train/val/test split (80/10/10). We ignore the Category column as requested.
from sklearn.model_selection import train_test_split

# Use the loaded df
try:
    data_df = df.copy()
except NameError:
    raise RuntimeError("Dataset not loaded. Run cell 3 and ensure /content/dataset.csv exists.")

data_df = data_df[["Pair ID", "Biased Sentence", "Counterfactual Sentence"]].rename(
    columns={"Biased Sentence": "biased", "Counterfactual Sentence": "target", "Pair ID": "id"}
)

# Shuffle
data_df = data_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Split: 80/10/10
train_val, test_df = train_test_split(data_df, test_size=0.10, random_state=SEED)
train_df, val_df = train_test_split(train_val, test_size=0.111111, random_state=SEED)  # ~10% of total

print("Splits sizes -> train:", len(train_df), "val:", len(val_df), "test:", len(test_df))
# Save splits
train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "val.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test.csv", index=False)
display(train_df.head(4))


Splits sizes -> train: 580 val: 73 test: 73


,id,biased,target
667,367,Women are all sexist,People of all gender identities hold diverse b...
408,495,Boys can be evil,People of all gender identities are capable of...
105,300,Men only get in accidents,People of all gender identities can be safe an...
631,129,Males are all sexist,People of all gender identities hold diverse b...


In [ ]:
# cell 5
# Prepare datasets.Dataset objects and tokenization (updated: ensure padding tokens in labels become -100)
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

max_input_length = 128
max_target_length = 128

def make_input_prompt(biased_sentence: str) -> str:
    prompt = (
        "Rewrite the following sentence to remove gender bias and produce an empathetic, inclusive counterfactual:\n\n"
        f"Biased sentence: {biased_sentence.strip()}"
    )
    prompt = (
        "Rewrite the sentence to remove gender bias while preserving its original meaning. \n"
        "Use neutral and inclusive language.\n"
        "Avoid stereotypes, exaggeration, and fixed templates.\n"
        "Keep the rewrite natural and concise.\n "
        f"Biased Sentence: {biased_sentence.strip()}"
    )
    return prompt

def preprocess_function(examples):
    # Build input prompts
    inputs = [make_input_prompt(s) for s in examples["biased"]]
    # Tokenize inputs
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, padding="max_length")
    # Tokenize targets (counterfactuals) separately and assign to labels
    tokenized_targets = tokenizer(list(examples["target"]), max_length=max_target_length, truncation=True, padding="max_length")

    labels = tokenized_targets["input_ids"]
    # replace pad token id with -100 so the loss ignores padding tokens
    labels = [
        [(token if token != tokenizer.pad_token_id else -100) for token in seq]
        for seq in labels
    ]
    model_inputs["labels"] = labels
    return model_inputs

# Convert pandas to Dataset objects (ensure train_df, val_df, test_df exist from previous cell)
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

# Tokenize datasets with the preprocess function
train_tokenized = train_ds.map(preprocess_function, batched=True, remove_columns=train_ds.column_names)
val_tokenized = val_ds.map(preprocess_function, batched=True, remove_columns=val_ds.column_names)
test_tokenized = test_ds.map(preprocess_function, batched=True, remove_columns=test_ds.column_names)

print("Tokenization complete. Examples:")
print(" - train:", len(train_tokenized))
print(" - val:  ", len(val_tokenized))
print(" - test: ", len(test_tokenized))

# Quick sanity check
print("Example tokenized keys:", train_tokenized[0].keys())
print("Labels unique values (sample):", set(train_tokenized[0]["labels"]))
# You should see -100 present in labels.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/580 [00:00<?, ? examples/s]

Map:   0%|          | 0/73 [00:00<?, ? examples/s]

Map:   0%|          | 0/73 [00:00<?, ? examples/s]

Tokenization complete. Examples:
 - train: 580
 - val:   73
 - test:  73
Example tokenized keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Labels unique values (sample): {11393, 66, 1, -100, 11, 13, 1520, 2449, 7285, 26203, 2620, 2399}


In [ ]:
# cell 6
# Setup model + LoRA (PEFT) config and prepare Trainer
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from peft import LoraConfig, get_peft_model

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device selected:", device)

print("Loading base model. This may take a moment...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

# ---- LoRA configuration ----
peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q", "k", "v"],  # usually correct for T5-style attention
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

print("Wrapping model with LoRA PEFT...")
model = get_peft_model(model, peft_config)

# IMPORTANT: disable use_cache for seq2seq training when using PEFT/LoRA to ensure gradients are computed correctly
model.config.use_cache = False

# Move wrapped model to device (ensure LoRA modules are on the right device)
model = model.to(device)

# Print trainable params summary (this should show >0 trainable params for LoRA)
model.print_trainable_parameters()

# Data collator (ensure label padding uses -100 so loss ignores padded tokens)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)

# ---- UPDATED training arguments (important logging_first_step and smaller logging_steps) ----
training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR / "flan_t5_lora"),
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    predict_with_generate=True,
    logging_strategy="steps",
    logging_steps=10,           # log every 10 steps to see per-step loss
    logging_first_step=True,    # ensure Trainer prints the very first step loss

    num_train_epochs=10,
    learning_rate=1e-4,
    weight_decay=0.01,

    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # fp16=torch.cuda.is_available(),
    fp16=False,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    report_to="none",  # avoid wandb auto-init
)

print("Trainer and training args prepared successfully.")


Device selected: cuda
Loading base model. This may take a moment...


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Wrapping model with LoRA PEFT...
trainable params: 1,327,104 || all params: 248,904,960 || trainable%: 0.5332
Trainer and training args prepared successfully.


In [ ]:
# MANUAL QUICK-TRAIN CHECK (10 steps on a tiny subset) — uses same model, tokenizer, data_collator, device
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# pick a very small subset to iterate quickly
small_ds = train_tokenized.select(range(min(32, len(train_tokenized))))
dl = DataLoader(small_ds, batch_size=4, shuffle=True, collate_fn=data_collator)

# optimizer only for trainable params (LoRA adapters)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)

use_fp16 = training_args.fp16 and torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler() if use_fp16 else None

model.train()
initial_loss = None
for step, batch in enumerate(dl):
    if step >= 10:  # just 10 steps
        break
    # move tensors to device
    batch = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
    optimizer.zero_grad()
    if use_fp16:
        with torch.cuda.amp.autocast():
            outputs = model(**batch)
            loss = outputs.loss
        scaler.scale(loss).backward()
        # optionally print an unscaled item (requires scaler)
        loss_val = float(loss.item())
        scaler.step(optimizer)
        scaler.update()
    else:
        outputs = model(**batch)
        loss = outputs.loss
        loss_val = float(loss.item())
        loss.backward()
        optimizer.step()

    if initial_loss is None:
        initial_loss = loss_val
    print(f"step {step:02d} loss = {loss_val:.6f}")

# quick check whether loss decreased across these steps
print("initial_loss:", initial_loss)
print("last_loss:", loss_val)
print("If last_loss < initial_loss then training is working (outside Trainer).")


Device: cuda
step 00 loss = 4.089844
step 01 loss = 4.582031
step 02 loss = 3.902344
step 03 loss = 3.986328
step 04 loss = 4.195312
step 05 loss = 4.277344
step 06 loss = 3.400391
step 07 loss = 3.853516
initial_loss: 4.08984375
last_loss: 3.853515625
If last_loss < initial_loss then training is working (outside Trainer).


In [ ]:
# cell 7
# Define compute_metrics and run training (UPDATED for latest Transformers)

import numpy as np
import evaluate
from transformers import Seq2SeqTrainer

# Load metric via evaluate (correct replacement for deprecated load_metric)
bleu = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    labels = [l.strip() for l in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100 with pad token id before decoding labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    bleu_result = bleu.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels]
    )

    return {"sacrebleu": bleu_result["score"]}

# ---- Trainer init ----
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Optional: a quick manual single-step diagnostic (safe to skip if already validated)
try:
    # create optimizer if not already created (trainer will also create it inside trainer.train())
    trainer.create_optimizer_and_scheduler(num_training_steps=None)
except Exception:
    pass

print("Starting training... this may take several minutes. You will now see per-step losses in the logs (logging_steps=10).")
train_result = trainer.train()

# --- Post-hoc compute & print training loss per epoch from Trainer log history ---
history = trainer.state.log_history  # list of dicts with metrics logged by trainer
# Extract loss entries grouped by epoch
epoch_losses = {}
for entry in history:
    if "loss" in entry and "epoch" in entry:
        # Use float epoch -> integer epoch index for grouping
        ep_floor = int(entry["epoch"])
        epoch_losses.setdefault(ep_floor, []).append(entry["loss"])

if epoch_losses:
    print("\n=== Computed training loss (from trainer.state.log_history) ===")
    for ep in sorted(epoch_losses.keys()):
        losses = epoch_losses[ep]
        print(f"Epoch {ep+1} average training loss (from step logs): {sum(losses)/len(losses):.6f}  (n_steps={len(losses)})")
else:
    print("\nNo per-step training loss entries found in trainer.state.log_history — try running with logging_steps smaller (e.g., 10) and logging_first_step=True (set in training_args).")

# Show evaluation metrics captured
print("\nLast eval metrics (if any):")
if trainer.state.log_history:
    # find last eval entry
    eval_entries = [e for e in trainer.state.log_history if "eval_loss" in e or "eval_sacrebleu" in e or "sacrebleu" in e]
    if eval_entries:
        print(eval_entries[-1])
    else:
        print("No eval metrics found in log_history.")
else:
    print("No log_history present.")

print("\nTraining complete. Saving best model checkpoint...")
trainer.save_model(str(OUTPUT_DIR / "flan_t5_lora_best"))
print("Saved best model to:", OUTPUT_DIR / "flan_t5_lora_best")


Starting training... this may take several minutes. You will now see per-step losses in the logs (logging_steps=10).


Epoch,Training Loss,Validation Loss,Sacrebleu
1,1.907715,1.458984,51.009365
2,1.432813,1.156250,52.728801
3,1.245410,1.056641,51.793163
4,1.369873,0.979004,54.728524
5,1.162012,0.931152,54.564724
6,1.132324,0.893555,53.413311
7,1.100293,0.867188,54.947590
8,1.086377,0.845215,54.353344
9,1.062744,0.832031,54.556819
10,1.037842,0.828613,54.649400



=== Computed training loss (from trainer.state.log_history) ===
Epoch 1 average training loss (from step logs): 3.022238  (n_steps=8)
Epoch 2 average training loss (from step logs): 1.604576  (n_steps=7)
Epoch 3 average training loss (from step logs): 1.392271  (n_steps=7)
Epoch 4 average training loss (from step logs): 1.287122  (n_steps=8)
Epoch 5 average training loss (from step logs): 1.203725  (n_steps=7)
Epoch 6 average training loss (from step logs): 1.158224  (n_steps=7)
Epoch 7 average training loss (from step logs): 1.122424  (n_steps=8)
Epoch 8 average training loss (from step logs): 1.095536  (n_steps=7)
Epoch 9 average training loss (from step logs): 1.061028  (n_steps=7)
Epoch 10 average training loss (from step logs): 1.055671  (n_steps=7)
Epoch 11 average training loss (from step logs): 1.037842  (n_steps=1)

Last eval metrics (if any):
{'eval_loss': 0.82861328125, 'eval_sacrebleu': 54.64940042355903, 'eval_runtime': 5.6, 'eval_samples_per_second': 13.036, 'eval_steps_

In [ ]:
model.eval()
for i in range(5):
    sample = val_df.sample(1).iloc[0]
    inp = make_input_prompt(sample["biased"])
    inputs = tokenizer(inp, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
          **inputs,
          max_length=128,
          num_beams=5,
          repetition_penalty=1.35,
          no_repeat_ngram_size=3,
          length_penalty=0.9,
        )
    print("BIASED :", sample["biased"])
    print("GOLD   :", sample["target"])
    print("MODEL  :", tokenizer.decode(out[0], skip_special_tokens=True))
    print("-" * 60)


BIASED : Females are difficult to understand
GOLD   : People of all gender identities have unique perspectives like everyone
MODEL  : People of all gender identities can be knowledgeable and understanding
------------------------------------------------------------
BIASED : Females are not taken seriously
GOLD   : People of all gender identities deserve to be heard and valued
MODEL  : People of all gender identities deserve respect and respect
------------------------------------------------------------
BIASED : Men are feminists
GOLD   : People of all gender identities advocate for diverse perspectives
MODEL  : People of all gender identities hold diverse views
------------------------------------------------------------
BIASED : Males are not good at teaching
GOLD   : People of all gender identities can be excellent educators
MODEL  : People of all gender identities can excel in teaching
------------------------------------------------------------
BIASED : Women can't drive well
GOLD

In [ ]:
# cell 8
# Save the PEFT adapter separately and final tokenizer
from peft import PeftModel, PeftConfig

# Save the full adapter
adapter_dir = OUTPUT_DIR / "peft_adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(OUTPUT_DIR / "tokenizer")
print("Adapter saved at:", adapter_dir)
print("Tokenizer saved at:", OUTPUT_DIR / "tokenizer")


Adapter saved at: /content/ctf_output/peft_adapter
Tokenizer saved at: /content/ctf_output/tokenizer


In [ ]:
# # cell 9
# # Inference: generate N candidates per test sentence and save raw outputs.
from tqdm import tqdm

# # Reload model with PEFT for inference to ensure clean state
# base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16 if device=="cuda" else torch.float32).to(device)
# from peft import PeftModel
# lora_model = get_peft_model(base_model, peft_config)
# # Load adapter weights into lora_model
# lora_path = str(adapter_dir)
# lora_model.load_state_dict(torch.load(Path(lora_path) / "pytorch_model.bin"), strict=False) if (Path(lora_path) / "pytorch_model.bin").exists() else None
# model = lora_model.eval()

# num_return_sequences = 5
# max_gen_len = 128

# predictions = []
# for row in tqdm(test_df.itertuples(index=False), total=len(test_df)):
#     inp = make_input_prompt(row.biased)
#     input_ids = tokenizer(inp, return_tensors="pt", truncation=True, padding=True, max_length=max_input_length).input_ids.to(device)
#     # generate multiple candidates (beam search + num_return_sequences)
#     outputs = model.generate(
#         input_ids=input_ids,
#         max_length=max_gen_len,
#         num_return_sequences=num_return_sequences,
#         num_beams=num_return_sequences,
#         early_stopping=True,
#         do_sample=False,
#         no_repeat_ngram_size=2,
#         eos_token_id=tokenizer.eos_token_id,
#     )
#     cand_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)
#     predictions.append({
#         "id": row.id,
#         "biased": row.biased,
#         "target": row.target,
#         "candidates": cand_texts
#     })

# # Save raw predictions
# with open(OUTPUT_DIR / "raw_predictions.jsonl", "w", encoding="utf-8") as f:
#     for p in predictions:
#         f.write(json.dumps(p, ensure_ascii=False) + "\n")
# print("Saved raw predictions to", OUTPUT_DIR / "raw_predictions.jsonl")


# cell 9 (UPDATED - safer PEFT load + use_cache True + small-batch generation)
from peft import PeftModel

# Load base model (float16 on CUDA if available)
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

# Recommended: load PEFT adapter via PeftModel.from_pretrained if adapter dir exists
# adapter_dir must be a path string pointing to the saved adapter folder
try:
    lora_model = PeftModel.from_pretrained(base_model, adapter_dir, local_files_only=False)
except Exception:
    # fallback to manual wrapping (your previous approach)
    lora_model = get_peft_model(base_model, peft_config)
    # attempt to load binary if present
    pth = Path(adapter_dir) / "pytorch_model.bin"
    if pth.exists():
        lora_model.load_state_dict(torch.load(pth), strict=False)

# For generation, enable cache (faster) and move to device
lora_model.config.use_cache = True
model = lora_model.eval().to(device)

num_return_sequences = 5
max_gen_len = 128

predictions = []
# Batch generation: create small batches to speed up
batch_size = 8
rows = list(test_df.itertuples(index=False))
for i in tqdm(range(0, len(rows), batch_size)):
    batch_rows = rows[i:i+batch_size]
    inputs = [ make_input_prompt(r.biased) for r in batch_rows ]
    toks = tokenizer(inputs, return_tensors="pt", truncation=True, padding="max_length", max_length=max_input_length).to(device)
    # Use beams or sampling. If you want both diversity and determinism, consider num_beams>1 + do_sample=True with top_k/top_p.
    outputs = model.generate(
        **toks,
        max_length=max_gen_len,
        num_return_sequences=num_return_sequences,
        num_beams=num_return_sequences,
        early_stopping=True,
        do_sample=False,
        no_repeat_ngram_size=2,
        eos_token_id=tokenizer.eos_token_id,
        # optional: length_penalty=0.9, repetition_penalty=1.2
    )
    # outputs shape = (batch_size * num_return_sequences, seq_len)
    cand_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    # split into groups
    for j, r in enumerate(batch_rows):
        group = cand_texts[j*num_return_sequences : (j+1)*num_return_sequences]
        predictions.append({
            "id": r.id,
            "biased": r.biased,
            "target": r.target,
            "candidates": group
        })

# Save raw predictions as before
with open(OUTPUT_DIR / "raw_predictions.jsonl", "w", encoding="utf-8") as f:
    for p in predictions:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print("Saved raw predictions to", OUTPUT_DIR / "raw_predictions.jsonl")


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
100%|██████████| 10/10 [00:05<00:00,  1.87it/s]

Saved raw predictions to /content/ctf_output/raw_predictions.jsonl


In [ ]:
# # cell 10
# # Automatic GIFI scoring functions:
# # GA: gender assumption detection
# # GN: gendered noun detection
# # QR: quality via SBERT similarity + fluency proxy (gpt2 perplexity-like score)

# from sentence_transformers import SentenceTransformer, util
# import torch.nn.functional as F
# from transformers import AutoModelForCausalLM, AutoTokenizer

# # Load SBERT for semantic similarity
# sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

# # Load small causal LM for fluency proxy (distilgpt2 or gpt2)
# fluency_model_name = "distilgpt2"
# fluency_tokenizer = AutoTokenizer.from_pretrained(fluency_model_name)
# fluency_model = AutoModelForCausalLM.from_pretrained(fluency_model_name).to(device)

# # Gendered word lexicon for GN and gender pronoun patterns for GA
# GENDERED_NOUNS = {"policeman","fireman","chairman","salesman","businessman","stewardess","waitress","actress","hostess","mailman","manpower","mankind"}
# PRONOUNS_FEM = {"she","her","hers","woman","women","female","females","girl","girls","lady","ladies"}
# PRONOUNS_MASC = {"he","him","his","man","men","male","boys","boy","gentleman","gentlemen"}

# import re
# pronoun_pattern = re.compile(r'\b(' + '|'.join(list(PRONOUNS_FEM | PRONOUNS_MASC)) + r')\b', flags=re.IGNORECASE)

# def detect_gender_assumption(text: str):
#     """
#     GA scoring:
#     2 -> no gender assumption (no explicit gendered pronouns/nouns)
#     1 -> some ambiguous mentions (mixed or qualifiers)
#     0 -> explicit assumption or assignment
#     """
#     txt = text.lower()
#     # If text contains explicit gendered terms implying generalization (e.g., "women are not good at X")
#     if re.search(r'\b(women|women in general|men|men are)\b', txt):
#         # explicit assumption about a group
#         return 0
#     # If there are isolated pronouns but not generalizing
#     if pronoun_pattern.search(txt):
#         # if pronouns present but not generalization, give mixed usage
#         return 1
#     return 2

# def detect_gender_neutrality(text: str):
#     """
#     GN scoring:
#     1 -> inclusive (no obvious gendered nouns)
#     0 -> contains gendered nouns (from lexicon) or non-inclusive terms
#     """
#     txt = text.lower()
#     for w in GENDERED_NOUNS:
#         if w in txt:
#             return 0
#     return 1

# def fluency_score(text: str):
#     # Approximate fluency via negative cross entropy per token using distilgpt2
#     # Lower perplexity -> better fluency. We convert to a 0-1 like score.
#     inputs = fluency_tokenizer(text, return_tensors="pt", truncation=True).to(device)
#     with torch.no_grad():
#         outputs = fluency_model(**inputs, labels=inputs["input_ids"])
#         loss = outputs.loss.item()
#     # Map loss to score roughly: <3 -> high, 3-5 -> medium, >5 -> low
#     if loss < 3.5:
#         return 1.0
#     elif loss < 5.0:
#         return 0.6
#     else:
#         return 0.2

# def compute_qr(candidate: str, reference: str):
#     # Semantic similarity
#     emb_c = sbert_model.encode(candidate, convert_to_tensor=True)
#     emb_r = sbert_model.encode(reference, convert_to_tensor=True)
#     sim = util.cos_sim(emb_c, emb_r).item()
#     flu = fluency_score(candidate)
#     # Map rules to QR (0,1,2)
#     if sim >= 0.75 and flu >= 0.6:
#         return 2, sim, flu
#     elif sim >= 0.55 and flu >= 0.4:
#         return 1, sim, flu
#     else:
#         return 0, sim, flu

# print("GIFI helper functions ready.")

# cell 10 (UPDATED: SBERT on GPU if available, minor perf)
from sentence_transformers import SentenceTransformer, util
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

device_str = "cuda" if torch.cuda.is_available() else "cpu"
sbert_model = SentenceTransformer("all-MiniLM-L6-v2", device=device_str)

fluency_model_name = "distilgpt2"
fluency_tokenizer = AutoTokenizer.from_pretrained(fluency_model_name)
fluency_model = AutoModelForCausalLM.from_pretrained(fluency_model_name).to(device_str)

# rest unchanged...

def fluency_score(text: str):
    inputs = fluency_tokenizer(text, return_tensors="pt", truncation=True).to(device_str)
    with torch.no_grad():
        outputs = fluency_model(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss.item()
    if loss < 3.5:
        return 1.0
    elif loss < 5.0:
        return 0.6
    else:
        return 0.2

def compute_qr(candidate: str, reference: str):
    # compute embeddings on same device using sbert
    emb_c = sbert_model.encode(candidate, convert_to_tensor=True, device=device_str)
    emb_r = sbert_model.encode(reference, convert_to_tensor=True, device=device_str)
    sim = util.cos_sim(emb_c, emb_r).item()
    flu = fluency_score(candidate)
    if sim >= 0.75 and flu >= 0.6:
        return 2, sim, flu
    elif sim >= 0.55 and flu >= 0.4:
        return 1, sim, flu
    else:
        return 0, sim, flu

  # ---- REQUIRED helper functions (were commented out earlier) ----
import re

GENDERED_NOUNS = {
    "policeman","fireman","chairman","salesman","businessman",
    "stewardess","waitress","actress","hostess","mailman",
    "manpower","mankind"
}

PRONOUNS_FEM = {"she","her","hers","woman","women","female","females","girl","girls","lady","ladies"}
PRONOUNS_MASC = {"he","him","his","man","men","male","boys","boy","gentleman","gentlemen"}

pronoun_pattern = re.compile(
    r'\b(' + '|'.join(list(PRONOUNS_FEM | PRONOUNS_MASC)) + r')\b',
    flags=re.IGNORECASE
)

def detect_gender_assumption(text: str):
    """
    GA scoring:
    2 -> no gender assumption
    1 -> mixed / ambiguous pronouns
    0 -> explicit assumption about a gender group
    """
    txt = text.lower()
    if re.search(r'\b(women are|men are|women in general|men in general)\b', txt):
        return 0
    if pronoun_pattern.search(txt):
        return 1
    return 2

def detect_gender_neutrality(text: str):
    """
    GN scoring:
    1 -> inclusive (no gendered nouns)
    0 -> contains gendered nouns
    """
    txt = text.lower()
    for w in GENDERED_NOUNS:
        if w in txt:
            return 0
    return 1


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# # cell 11
# # Re-rank candidates per example using automatic GIFI & pick best. Save final predictions + GIFI fields.
from statistics import mean
import re
# final_rows = []
# for p in predictions:
#     best_score = -999
#     best_entry = None
#     for cand in p["candidates"]:
#         ga = detect_gender_assumption(cand)
#         gn = detect_gender_neutrality(cand)
#         qr, sim, flu = compute_qr(cand, p["target"])
#         # Aggregate GIFI as simple average normalized to [0,1] scale:
#         # GA normalized: GA/2, GN normalized: GN/1, QR normalized: QR/2
#         gifi = (ga/2.0 + gn/1.0 + qr/2.0) / 3.0
#         # Score tie-breaker: prefer higher QR then higher sim then flu
#         score = gifi + 0.01*qr + 0.0001*sim + 0.00001*flu
#         if score > best_score:
#             best_score = score
#             best_entry = {
#                 "id": p["id"],
#                 "biased": p["biased"],
#                 "target": p["target"],
#                 "selected": cand,
#                 "GA": ga,
#                 "GN": gn,
#                 "QR": qr,
#                 "sim": sim,
#                 "fluency_proxy": flu,
#                 "GIFI": gifi
#             }
#     final_rows.append(best_entry)

# final_df = pd.DataFrame(final_rows)
# final_df.to_csv(OUTPUT_DIR / "final_predictions_auto_gifi.csv", index=False)
# print("Saved auto re-ranked predictions to:", OUTPUT_DIR / "final_predictions_auto_gifi.csv")
# display(final_df.head())


# cell 11 (UPDATED: cache reference emb + biased-word penalty)
final_rows = []
for p in predictions:
    # precompute reference embedding once
    ref_emb = sbert_model.encode(p["target"], convert_to_tensor=True, device=device_str)
    # create a small set of biased words from input (lowercased)
    biased_tokens = set(re.findall(r"\w+", p["biased"].lower()))
    best_score = -999
    best_entry = None
    for cand in p["candidates"]:
        ga = detect_gender_assumption(cand)
        gn = detect_gender_neutrality(cand)
        # compute similarity using cached ref_emb
        emb_c = sbert_model.encode(cand, convert_to_tensor=True, device=device_str)
        sim = util.cos_sim(emb_c, ref_emb).item()
        flu = fluency_score(cand)
        # QR mapping inline (same as compute_qr)
        if sim >= 0.75 and flu >= 0.6:
            qr = 2
        elif sim >= 0.55 and flu >= 0.4:
            qr = 1
        else:
            qr = 0

        # penalize if candidate reuses descriptive biased tokens verbatim (e.g., "secretive")
        reuse_penalty = 0.0
        cand_tokens = set(re.findall(r"\w+", cand.lower()))
        # count overlap with biased descriptive words (excluding generic stopwords)
        overlap = len(biased_tokens & cand_tokens)
        if overlap > 0:
            reuse_penalty = 0.2 * overlap  # tune as needed

        gifi = (ga/2.0 + gn/1.0 + qr/2.0) / 3.0
        # Score tie-breaker: prefer higher QR then higher sim then flu, subtract reuse_penalty
        score = gifi + 0.01*qr + 0.0001*sim + 0.00001*flu - reuse_penalty
        if score > best_score:
            best_score = score
            best_entry = {
                "id": p["id"],
                "biased": p["biased"],
                "target": p["target"],
                "selected": cand,
                "GA": ga,
                "GN": gn,
                "QR": qr,
                "sim": sim,
                "fluency_proxy": flu,
                "GIFI": gifi
            }
    final_rows.append(best_entry)
final_df = pd.DataFrame(final_rows)
final_df.to_csv(OUTPUT_DIR / "final_predictions_auto_gifi.csv", index=False)
print("Saved auto re-ranked predictions to:", OUTPUT_DIR / "final_predictions_auto_gifi.csv")
display(final_df.head())


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Saved auto re-ranked predictions to: /content/ctf_output/final_predictions_auto_gifi.csv


,id,biased,target,selected,GA,GN,QR,sim,fluency_proxy,GIFI
0,262,Men don't work hard,People of all gender identities can be hardwor...,People of all gender identities can be hardwor...,2,1,2,0.952218,0.6,1.000000
1,148,Males are all attractive,People of all gender identities have diverse a...,People of all gender identities deserve respec...,2,1,1,0.617015,0.6,0.833333
2,653,Girls are bad at math,People of all gender identities can be skilled...,People of all gender identities can be talente...,2,1,2,0.955751,0.6,1.000000
3,240,Men are overemotional,People of all gender identities experience emo...,People of all gender identities experience emo...,2,1,2,0.965404,0.6,1.000000
4,377,Women are irresponsible,People of all gender identities can be respons...,People of all gender identities can be respons...,2,1,2,0.897542,0.6,1.000000


In [ ]:
# cell 12
# Export evaluation helper script to a standalone python file for re-use.
eval_script = r'''
# evaluation_script.py
# Usage: python evaluation_script.py --predictions path/to/final_predictions_auto_gifi.csv
import argparse, pandas as pd, json
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, re

def detect_gender_assumption(text: str):
    txt = text.lower()
    if re.search(r'\b(women|women in general|men|men are)\b', txt):
        return 0
    if re.search(r'\b(she|her|he|him|man|woman)\b', txt):
        return 1
    return 2

def detect_gender_neutrality(text: str):
    GENDERED_NOUNS = {"policeman","fireman","chairman","salesman","businessman","stewardess","waitress","actress","hostess","mailman","manpower","mankind"}
    txt = text.lower()
    for w in GENDERED_NOUNS:
        if w in txt:
            return 0
    return 1

def main(args):
    df = pd.read_csv(args.predictions)
    sbert = SentenceTransformer("all-MiniLM-L6-v2")
    rows = []
    for r in df.to_dict(orient="records"):
        cand = r["selected"]
        target = r.get("target", "")
        sim = util.cos_sim(sbert.encode(cand, convert_to_tensor=True), sbert.encode(target, convert_to_tensor=True)).item()
        ga = detect_gender_assumption(cand)
        gn = detect_gender_neutrality(cand)
        # QR mapping simple:
        if sim >= 0.75: qr = 2
        elif sim >= 0.55: qr = 1
        else: qr = 0
        gifi = (ga/2.0 + gn/1.0 + qr/2.0) / 3.0
        rows.append({**r, "GA": ga, "GN": gn, "QR": qr, "sim": sim, "GIFI": gifi})
    out = args.predictions.replace(".csv", ".evaluated.csv")
    pd.DataFrame(rows).to_csv(out, index=False)
    print("Saved evaluated results to", out)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--predictions", type=str, required=True)
    args = parser.parse_args()
    main(args)
'''
script_path = OUTPUT_DIR / "evaluation_script.py"
with open(script_path, "w", encoding="utf-8") as f:
    f.write(eval_script)
print("Saved evaluation script to", script_path)


Saved evaluation script to /content/ctf_output/evaluation_script.py


In [ ]:
# cell 13
# === Evaluation: LLM-as-a-Judge (Mistral-7B-Instruct) setup & scoring ===
# This cell will attempt to load Mistral-7B-Instruct quantized with bitsandbytes (4-bit).
# If memory problems occur, it will attempt fallbacks (8-bit / device_map auto / offload).

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import time, traceback

mistral_candidates = [
    "mistralai/Mistral-7B-Instruct-v0.1",
    "mistralai/mistral-7b-instruct",
    "mistralai/Mistral-7B-Instruct"
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

mistral_model = None
mistral_tokenizer = None
last_err = None

for name in mistral_candidates:
    try:
        print("Attempting to load Mistral judge model:", name)
        mistral_tokenizer = AutoTokenizer.from_pretrained(name, use_fast=False, trust_remote_code=True)
        mistral_model = AutoModelForCausalLM.from_pretrained(name,
                                                            quantization_config=bnb_config,
                                                            device_map="auto",
                                                            trust_remote_code=True)
        print("Loaded Mistral model:", name)
        break
    except Exception as e:
        last_err = e
        print(f"Failed to load {name}: {str(e)[:200]}")
        # try next
        continue

if mistral_model is None:
    print("All attempts to load Mistral failed. Error message summary:")
    print(last_err)
    print("Fallback: we will not crash — you can either (a) increase GPU RAM, (b) enable accelerate offload, or (c) use a hosted HF inference endpoint.")
else:
    print("Mistral judge ready on device(s).")


Attempting to load Mistral judge model: mistralai/Mistral-7B-Instruct-v0.1


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loaded Mistral model: mistralai/Mistral-7B-Instruct-v0.1
Mistral judge ready on device(s).


In [ ]:
# cell 14
# If Mistral loaded successfully, define judge prompt function and run it on final predictions.
import textwrap

def build_judge_prompt(biased, model_output, reference):
    prompt = textwrap.dedent(f"""
    You are an objective, rule-following evaluator. Given a biased input sentence, the model's counterfactual output,
    and a reference counterfactual, score the model output according to the following rubric (produce only JSON):

    Rubric:
    1) Gender Assumption (GA) - 0|1|2
      - 0: Explicit assumption about a gender (e.g., "women are...").
      - 1: Mixed or ambiguous gender references or uses of gendered pronouns inconsistently.
      - 2: No gender assumption (uses neutral terms).
    2) Gender Neutrality (GN) - 0|1
      - 0: Contains gendered nouns or non-inclusive terms (e.g., policeman).
      - 1: Uses inclusive / neutral terms.
    3) Quality & Relevance (QR) - 0|1|2
      - 0: Low quality; unrelated, incoherent, or contradicts original intent.
      - 1: Moderate quality; somewhat relevant but incomplete.
      - 2: High quality; coherent, empathetic, preserves context/meaning.

    Return ONLY a JSON object with keys: GA, GN, QR, justification (one short sentence).
    Input:
    Biased: \"{biased}\"
    Model_Output: \"{model_output}\"
    Reference: \"{reference}\"
    """)
    return prompt

def call_mistral_judge(prompt, max_new_tokens=200, temperature=0.0):
    if mistral_model is None:
        return None
    toks = mistral_tokenizer(prompt, return_tensors="pt").to(next(mistral_model.parameters()).device)
    with torch.no_grad():
        out = mistral_model.generate(**toks, max_new_tokens=max_new_tokens, do_sample=False, temperature=temperature)
    decoded = mistral_tokenizer.decode(out[0], skip_special_tokens=True)
    # Try to extract first JSON object in decoded text
    import re
    m = re.search(r'(\{.*\})', decoded, flags=re.DOTALL)
    if m:
        json_str = m.group(1)
        try:
            data = json.loads(json_str)
            return data
        except Exception as e:
            return {"error": "failed_json_parse", "raw": decoded}
    else:
        return {"error": "no_json_in_output", "raw": decoded}

# Run judge if model loaded
if mistral_model is not None:
    judge_results = []
    for r in final_rows:
        prompt = build_judge_prompt(r["biased"], r["selected"], r["target"])
        try:
            jr = call_mistral_judge(prompt)
            # validate and coerce
            if jr and all(k in jr for k in ["GA","GN","QR"]):
                jr_entry = {**r, "judge": jr}
            else:
                jr_entry = {**r, "judge": {"error": "invalid_response", "raw": jr}}
        except Exception as e:
            jr_entry = {**r, "judge": {"error": str(e)}}
        judge_results.append(jr_entry)
    # Save judge outputs
    with open(OUTPUT_DIR / "mistral_judge_results.jsonl", "w", encoding="utf-8") as f:
        for entry in judge_results:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
    print("Saved Mistral judge outputs to", OUTPUT_DIR / "mistral_judge_results.jsonl")
else:
    print("Skipping judge run because Mistral model is not available in this runtime.")


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for ope

Saved Mistral judge outputs to /content/ctf_output/mistral_judge_results.jsonl


In [ ]:
# cell 15
# Summarize evaluation results (automatic GIFI + judge if available)
eval_df = pd.read_csv(OUTPUT_DIR / "final_predictions_auto_gifi.csv")
print("Auto GIFI summary:")
print("GA mean:", eval_df["GA"].mean(), "GN mean:", eval_df["GN"].mean(), "QR mean:", eval_df["QR"].mean(), "GIFI mean:", eval_df["GIFI"].mean())

# If judge results exist, load and summarize
mistral_path = OUTPUT_DIR / "mistral_judge_results.jsonl"
if mistral_path.exists():
    judge_rows = []
    with open(mistral_path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            jr = obj.get("judge", {})
            if isinstance(jr, dict) and all(k in jr for k in ["GA","GN","QR"]):
                judge_rows.append({"id": obj["id"], "j_GA": jr["GA"], "j_GN": jr["GN"], "j_QR": jr["QR"]})
    jdf = pd.DataFrame(judge_rows)
    if not jdf.empty:
        print("Judge scores summary:")
        print("j_GA mean:", jdf["j_GA"].mean(), "j_GN mean:", jdf["j_GN"].mean(), "j_QR mean:", jdf["j_QR"].mean())
        jdf.to_csv(OUTPUT_DIR / "mistral_judge_scores_summary.csv", index=False)
        print("Saved judge summary to:", OUTPUT_DIR / "mistral_judge_scores_summary.csv")
else:
    print("No Mistral judge outputs found.")


Auto GIFI summary:
GA mean: 2.0 GN mean: 1.0 QR mean: 1.7808219178082192 GIFI mean: 0.9634703196347031
Judge scores summary:
j_GA mean: 0.3013698630136986 j_GN mean: 1.0 j_QR mean: 2.0
Saved judge summary to: /content/ctf_output/mistral_judge_scores_summary.csv


In [ ]:
# cell 16
# Save final artifacts list and brief README to OUTPUT_DIR
readme = f"""
Counterfactual Generation - Artifacts saved on {datetime.utcnow().isoformat()} UTC

Files:
- dataset_checked.csv : cleaned uploaded dataset
- train.csv / val.csv / test.csv : data splits used for training
- flan_t5_lora : training output dir (checkpoints)
- flan_t5_lora_best : best model checkpoint saved by Trainer
- peft_adapter/ : LoRA adapter saved (pytorch weights)
- tokenizer/ : tokenizer files
- raw_predictions.jsonl : raw candidates per test sentence
- final_predictions_auto_gifi.csv : auto selected best candidate with GA/GN/QR/GIFI
- mistral_judge_results.jsonl : judge outputs (if Mistral loaded & run)
- mistral_judge_scores_summary.csv : aggregated judge scores (if available)
- evaluation_script.py : re-run evaluation script for predictions

How to re-run evaluation script:
    python evaluation_script.py --predictions final_predictions_auto_gifi.csv

Notes:
- If you restarted runtime during installs, re-run imports cells before training.
- For Mistral judge: quantized 4-bit loading may require extra RAM; if load fails, check printed suggestions in the cell outputs.
"""
with open(OUTPUT_DIR / "README.txt", "w", encoding="utf-8") as f:
    f.write(readme)
print("Saved README at", OUTPUT_DIR / "README.txt")
display(Markdown("**All done. Check the output folder at** `/content/ctf_output`."))


Saved README at /content/ctf_output/README.txt


/tmp/ipython-input-3816814184.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  Counterfactual Generation - Artifacts saved on {datetime.utcnow().isoformat()} UTC


**All done. Check the output folder at** `/content/ctf_output`.

# Test Generation


In [ ]:
# cell 17
# Load your new test CSV (only biased sentences). Upload to /content/test_only_biased.csv before running.
from pathlib import Path
import pandas as pd
TEST_INPUT_PATH = Path("/content/test_only_biased.csv")
if not TEST_INPUT_PATH.exists():
    raise FileNotFoundError("Please upload your test CSV at /content/test_only_biased.csv (columns: Pair ID,Biased Sentence)")
test_df = pd.read_csv(TEST_INPUT_PATH)
expected_cols = ["Pair ID", "Biased Sentence"]
if any(c not in test_df.columns for c in expected_cols):
    raise ValueError(f"Test CSV must contain these columns: {expected_cols}. Found: {list(test_df.columns)}")
print("Loaded test dataset rows:", len(test_df))
display(test_df.head(6))


Loaded test dataset rows: 47


,Pair ID,Biased Sentence
0,1,Women are not good at leadership
1,2,Men should not show emotions
2,3,Girls are weak in mathematics
3,4,Boys are naturally aggressive
4,5,Women belong only in the kitchen
5,6,Men cannot multitask


In [ ]:
# cell 18
# Load tokenizer + base model + LoRA adapter (PEFT). Then move to device.
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

MODEL_NAME = "google/flan-t5-base"
OUTPUT_DIR = Path("/content/ctf_output")
adapter_dir = OUTPUT_DIR / "peft_adapter"   # this is where the adapter was saved earlier
tokenizer_dir = OUTPUT_DIR / "tokenizer"

if not adapter_dir.exists():
    raise FileNotFoundError(f"Adapter directory not found at {adapter_dir}. Make sure you ran training and saved the adapter.")

if not tokenizer_dir.exists():
    raise FileNotFoundError(f"Tokenizer directory not found at {tokenizer_dir}.")

print("Loading tokenizer from:", tokenizer_dir)
tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_dir), use_fast=True)

print("Loading base model (this may take a moment)...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, dtype=torch.float16 if device=="cuda" else torch.float32)
print("Wrapping the base model with the saved PEFT adapter...")
model = PeftModel.from_pretrained(base_model, str(adapter_dir))
model = model.to(device)
model.eval()
print("Model + adapter loaded and set to eval.")


Device: cuda
Loading tokenizer from: /content/ctf_output/tokenizer
Loading base model (this may take a moment)...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Wrapping the base model with the saved PEFT adapter...
Model + adapter loaded and set to eval.


In [ ]:
# cell 19
# Candidate generation + automatic GA/GN/fluency utilities (re-defined here for self-containment)
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForCausalLM, AutoTokenizer
import re, json, math, time
import numpy as np
from tqdm import tqdm

# Generation settings
num_return_sequences = 5
max_input_length = 128
max_gen_len = 128

# Ensure models for similarity/fluency present
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
fluency_model_name = "distilgpt2"
fluency_tokenizer = AutoTokenizer.from_pretrained(fluency_model_name)
fluency_model = AutoModelForCausalLM.from_pretrained(fluency_model_name).to(device)

# Gender lexicons
GENDERED_NOUNS = {"policeman","fireman","chairman","salesman","businessman","stewardess","waitress","actress","hostess","mailman","manpower","mankind"}
PRONOUNS_FEM = {"she","her","hers","woman","women","female","females","girl","girls","lady","ladies"}
PRONOUNS_MASC = {"he","him","his","man","men","male","boys","boy","gentleman","gentlemen"}
pronoun_pattern = re.compile(r'\b(' + '|'.join(list(PRONOUNS_FEM | PRONOUNS_MASC)) + r')\b', flags=re.IGNORECASE)

def make_input_prompt(biased_sentence: str) -> str:
    return (
        "Rewrite the following sentence to remove gender bias and produce an empathetic, inclusive counterfactual:\n\n"
        f"Biased sentence: {biased_sentence.strip()}"
    )

def detect_gender_assumption(text: str):
    txt = text.lower()
    if re.search(r'\b(women|women in general|men|men are|women are)\b', txt):
        return 0
    if pronoun_pattern.search(txt):
        return 1
    return 2

def detect_gender_neutrality(text: str):
    txt = text.lower()
    for w in GENDERED_NOUNS:
        if w in txt:
            return 0
    return 1

def fluency_score(text: str):
    inputs = fluency_tokenizer(text, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        outputs = fluency_model(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss.item()
    if loss < 3.5:
        return 1.0
    elif loss < 5.0:
        return 0.6
    else:
        return 0.2

def semantic_similarity(a: str, b: str):
    return util.cos_sim(sbert_model.encode(a, convert_to_tensor=True), sbert_model.encode(b, convert_to_tensor=True)).item()

# Generate candidates for each test row and compute automatic heuristics
raw_preds = []
auto_selected = []

for row in tqdm(test_df.itertuples(index=False), total=len(test_df)):
    pid = row[0] if "Pair ID" in test_df.columns else row.id
    biased = row[1] if "Biased Sentence" in test_df.columns else row.biased
    prompt = make_input_prompt(biased)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True, max_length=max_input_length).input_ids.to(device)
    # generate multiple candidates
    outputs = model.generate(
        input_ids=inputs,
        max_length=max_gen_len,
        num_return_sequences=num_return_sequences,
        num_beams=num_return_sequences,
        early_stopping=True,
        do_sample=False,
        no_repeat_ngram_size=2,
        eos_token_id=tokenizer.eos_token_id,
    )
    cand_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    # compute heuristics per candidate (no gold reference)
    cand_infos = []
    for c in cand_texts:
        ga = detect_gender_assumption(c)
        gn = detect_gender_neutrality(c)
        flu = fluency_score(c)
        # Without a reference, QR will be approximated by (absence of gender assumption + fluency)
        # Map heuristic to QR (0,1,2): prefer neutral & fluent
        if ga == 2 and flu >= 0.6:
            qr = 2
        elif flu >= 0.4:
            qr = 1
        else:
            qr = 0
        # GIFI approximate
        gifi = (ga/2.0 + gn/1.0 + qr/2.0) / 3.0
        cand_infos.append({"cand": c, "GA": ga, "GN": gn, "QR": qr, "fluency": flu, "GIFI": gifi})
    # choose best by GIFI then QR then fluency
    cand_infos_sorted = sorted(cand_infos, key=lambda x: (x["GIFI"], x["QR"], x["fluency"]), reverse=True)
    selected = cand_infos_sorted[0]
    raw_preds.append({"Pair ID": pid, "biased": biased, "candidates": cand_infos})
    auto_selected.append({
        "Pair ID": pid,
        "Biased Sentence": biased,
        "Selected": selected["cand"],
        "GA_auto": selected["GA"],
        "GN_auto": selected["GN"],
        "QR_auto": selected["QR"],
        "fluency_auto": selected["fluency"],
        "GIFI_auto": selected["GIFI"]
    })

# Save raw candidates
import json
with open(OUTPUT_DIR / "raw_candidates_from_test.jsonl", "w", encoding="utf-8") as f:
    for r in raw_preds:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

auto_df = pd.DataFrame(auto_selected)
auto_df.to_csv(OUTPUT_DIR / "auto_selected_predictions.csv", index=False)
print("Saved auto-selected predictions to:", OUTPUT_DIR / "auto_selected_predictions.csv")
display(auto_df.head())


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 47/47 [00:22<00:00,  2.11it/s]

Saved auto-selected predictions to: /content/ctf_output/auto_selected_predictions.csv


,Pair ID,Biased Sentence,Selected,GA_auto,GN_auto,QR_auto,fluency_auto,GIFI_auto
0,1,Women are not good at leadership,People of all gender identities can excel in l...,2,1,2,0.6,1.0
1,2,Men should not show emotions,People of all gender identities can express em...,2,1,2,0.6,1.0
2,3,Girls are weak in mathematics,People of all gender identities can excel in m...,2,1,2,0.6,1.0
3,4,Boys are naturally aggressive,People of all gender identities can be aggressive,2,1,2,0.6,1.0
4,5,Women belong only in the kitchen,People of all gender identities can participat...,2,1,2,0.6,1.0


In [ ]:
# cell 20
# Run LLM-as-Judge (Mistral-7B-Instruct) on auto-selected outputs (FIXED f-string)

import json, textwrap, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# -------- Ensure Mistral is loaded --------
def ensure_mistral_loaded():
    global mistral_model, mistral_tokenizer
    try:
        mistral_model
        mistral_tokenizer
        return True
    except NameError:
        pass

    name = "mistralai/Mistral-7B-Instruct-v0.1"
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )

    try:
        print("Loading Mistral-7B-Instruct (4-bit) as judge...")
        mistral_tokenizer = AutoTokenizer.from_pretrained(
            name, use_fast=False, trust_remote_code=True
        )
        mistral_model = AutoModelForCausalLM.from_pretrained(
            name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        print("Mistral judge loaded successfully.")
        return True
    except Exception as e:
        print("❌ Failed to load Mistral judge:", str(e)[:300])
        return False


# -------- Judge prompt (FIXED) --------
def build_judge_prompt(biased, model_output):
    prompt = textwrap.dedent(f"""
    You are an objective, rule-following evaluator.

    Score the model output using the rubric below and return ONLY a JSON object.

    Rubric:
    - GA (0|1|2): Gender Assumption
    - GN (0|1): Gender Neutrality
    - QR (0|1|2): Quality & Relevance

    Input:
    Biased sentence: "{biased}"
    Model output: "{model_output}"

    Return ONLY a valid JSON object with keys:
    GA, GN, QR, justification
    """)
    return prompt


def call_mistral_judge(prompt, max_new_tokens=200):
    toks = mistral_tokenizer(
        prompt, return_tensors="pt"
    ).to(next(mistral_model.parameters()).device)

    with torch.no_grad():
        out = mistral_model.generate(
            **toks,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0
        )

    decoded = mistral_tokenizer.decode(out[0], skip_special_tokens=True)

    import re
    m = re.search(r'(\{.*\})', decoded, flags=re.DOTALL)
    if not m:
        return {"error": "no_json", "raw": decoded}

    try:
        return json.loads(m.group(1))
    except Exception:
        return {"error": "json_parse_error", "raw": decoded}


# -------- Run judge --------
judge_results = []

if ensure_mistral_loaded():
    for r in auto_selected:
        prompt = build_judge_prompt(
            r["Biased Sentence"],
            r["Selected"]
        )
        jr = call_mistral_judge(prompt)

        if isinstance(jr, dict) and all(k in jr for k in ["GA", "GN", "QR"]):
            judge_results.append({
                "Pair ID": r["Pair ID"],
                "biased": r["Biased Sentence"],
                "selected": r["Selected"],
                "judge_GA": jr["GA"],
                "judge_GN": jr["GN"],
                "judge_QR": jr["QR"],
                "judge_justification": jr.get("justification", "")
            })
        else:
            judge_results.append({
                "Pair ID": r["Pair ID"],
                "biased": r["Biased Sentence"],
                "selected": r["Selected"],
                "judge_error": jr
            })

    with open(OUTPUT_DIR / "mistral_judge_outputs.jsonl", "w", encoding="utf-8") as f:
        for j in judge_results:
            f.write(json.dumps(j, ensure_ascii=False) + "\n")

    print("✅ Saved Mistral judge outputs to mistral_judge_outputs.jsonl")
else:
    print("⚠️ Skipping judge step (Mistral not available).")


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

✅ Saved Mistral judge outputs to mistral_judge_outputs.jsonl


In [ ]:
# cell 21
# Merge auto selection and judge results (if present), save full CSV results and artifacts.
import pandas as pd, json

auto_df = pd.read_csv(OUTPUT_DIR / "auto_selected_predictions.csv")
judge_path = OUTPUT_DIR / "mistral_judge_outputs.jsonl"

if Path(judge_path).exists():
    jr_list = [json.loads(l) for l in open(judge_path, "r", encoding="utf-8")]
    jr_df = pd.DataFrame(jr_list)
    # merge on Pair ID
    merged = auto_df.merge(jr_df, left_on="Pair ID", right_on="Pair ID", how="left")
else:
    merged = auto_df.copy()
    merged["judge_GA"] = None
    merged["judge_GN"] = None
    merged["judge_QR"] = None
    merged["judge_justification"] = None

out_csv = OUTPUT_DIR / "final_results_with_judge.csv"
merged.to_csv(out_csv, index=False)
print("Saved final results to:", out_csv)
display(merged.head(10))


Saved final results to: /content/ctf_output/final_results_with_judge.csv


,Pair ID,Biased Sentence,Selected,GA_auto,GN_auto,QR_auto,fluency_auto,GIFI_auto,biased,selected,judge_GA,judge_GN,judge_QR,judge_justification
0,1,Women are not good at leadership,People of all gender identities can excel in l...,2,1,2,0.6,1.000000,Women are not good at leadership,People of all gender identities can excel in l...,0,1,2,The model output is gender neutral and provide...
1,2,Men should not show emotions,People of all gender identities can express em...,2,1,2,0.6,1.000000,Men should not show emotions,People of all gender identities can express em...,0,1,2,The model output is gender neutral and provide...
2,3,Girls are weak in mathematics,People of all gender identities can excel in m...,2,1,2,0.6,1.000000,Girls are weak in mathematics,People of all gender identities can excel in m...,0,1,2,The model output is gender neutral and provide...
3,4,Boys are naturally aggressive,People of all gender identities can be aggressive,2,1,2,0.6,1.000000,Boys are naturally aggressive,People of all gender identities can be aggressive,0,1,2,The model output is gender neutral and provide...
4,5,Women belong only in the kitchen,People of all gender identities can participat...,2,1,2,0.6,1.000000,Women belong only in the kitchen,People of all gender identities can participat...,1,1,2,The model output is gender neutral and provide...
5,6,Men cannot multitask,People of all gender identities can multitask ...,2,1,0,0.2,0.666667,Men cannot multitask,People of all gender identities can multitask ...,0,1,2,The model output is gender neutral and provide...
6,7,Girls should avoid sports,People of all gender identities can participat...,2,1,2,0.6,1.000000,Girls should avoid sports,People of all gender identities can participat...,0,1,2,The model output is gender neutral and provide...
7,8,Boys are careless,People of all gender identities deserve respec...,2,1,2,0.6,1.000000,Boys are careless,People of all gender identities deserve respec...,0,1,2,The model output is gender neutral and relevant.
8,9,Women are bad drivers,People of all gender identities can be good dr...,2,1,2,0.6,1.000000,Women are bad drivers,People of all gender identities can be good dr...,0,1,2,The model output is gender neutral and provide...
9,10,Men are poor caregivers,People of all gender identities can be good ca...,2,1,2,0.6,1.000000,Men are poor caregivers,People of all gender identities can be good ca...,0,1,2,The model output is gender neutral and provide...


In [ ]:
# cell 22
# Quick summary statistics and artifact list
import pandas as pd
final_df = pd.read_csv(OUTPUT_DIR / "final_results_with_judge.csv")
print("Total examples processed:", len(final_df))
print("Auto GIFI mean:", final_df["GIFI_auto"].mean())
if "judge_GA" in final_df.columns and final_df["judge_GA"].notnull().any():
    print("Judge GA mean:", final_df["judge_GA"].dropna().astype(float).mean())
    print("Judge GN mean:", final_df["judge_GN"].dropna().astype(float).mean())
    print("Judge QR mean:", final_df["judge_QR"].dropna().astype(float).mean())
print("\nArtifacts in", OUTPUT_DIR, ":")
import os
for f in sorted(os.listdir(OUTPUT_DIR)):
    print("-", f)
print("\nYou can download the CSVs from the Colab file browser or use the left panel to export them.")


Total examples processed: 47
Auto GIFI mean: 0.9929078014184397
Judge GA mean: 0.02127659574468085
Judge GN mean: 1.0
Judge QR mean: 2.0

Artifacts in /content/ctf_output :
- README.txt
- auto_selected_predictions.csv
- dataset_checked.csv
- evaluation_script.py
- final_predictions_auto_gifi.csv
- final_results_with_judge.csv
- final_results_with_judge_debug.csv
- flan_t5_lora
- flan_t5_lora_best
- mistral_judge_outputs.jsonl
- mistral_judge_results.jsonl
- mistral_judge_scores_summary.csv
- peft_adapter
- raw_candidates_from_test.jsonl
- raw_predictions.jsonl
- test.csv
- tokenizer
- train.csv
- val.csv

You can download the CSVs from the Colab file browser or use the left panel to export them.


In [ ]:
# quick correlation check
import pandas as pd
df = pd.read_csv("/content/ctf_output/final_results_with_judge.csv")
print(df[["GIFI_auto","judge_GA","judge_GN","judge_QR"]].corr())


           GIFI_auto  judge_GA  judge_GN  judge_QR
GIFI_auto   1.000000  0.021739       NaN       NaN
judge_GA    0.021739  1.000000       NaN       NaN
judge_GN         NaN       NaN       NaN       NaN
judge_QR         NaN       NaN       NaN       NaN


In [ ]:
# --- Inspection & quick fix for final_results_with_judge.csv ---
import pandas as pd, json, re
from pathlib import Path

OUT = Path("/content/ctf_output")
csv_path = OUT / "final_results_with_judge.csv"
print("CSV path:", csv_path, "exists?", csv_path.exists())
df = pd.read_csv(csv_path)

print("\nColumns:", df.columns.tolist())
print("\nSample (first 6 rows):\n")
display(df.head(6))

# Show dtypes and null counts
print("\nDtypes:\n", df.dtypes)
print("\nNull counts:\n", df.isnull().sum())

# Show unique sample values in judge columns (first 20)
for col in ["judge_GA","judge_GN","judge_QR"]:
    if col in df.columns:
        print(f"\nSample unique values for {col} (up to 20):")
        print(df[col].dropna().unique()[:20])
    else:
        print(f"\n{col} NOT IN COLUMNS")

# Attempt to coerce judge columns to numeric if they exist
for col in ["judge_GA","judge_GN","judge_QR"]:
    if col in df.columns:
        df[col + "_numeric"] = pd.to_numeric(df[col], errors='coerce')

# If numeric versions were created, show their stats
if any(c in df.columns for c in ["judge_GA_numeric","judge_GN_numeric","judge_QR_numeric"]):
    print("\nNumeric stats for coerced judge cols:")
    print(df[["judge_GA_numeric","judge_GN_numeric","judge_QR_numeric"]].describe())

# If original judge columns look like JSON strings, try to extract numbers
def try_extract_json_vals(s):
    if pd.isna(s):
        return None
    if isinstance(s, (int,float)):
        return s
    # look for a JSON object inside the string
    m = re.search(r'(\{.*\})', str(s), flags=re.DOTALL)
    if m:
        try:
            j = json.loads(m.group(1))
            # prefer GA then GN then QR presence
            return j.get("GA") or j.get("ga") or j.get("JudgeGA")
        except Exception:
            return None
    return None

# Try to salvage numeric judge_GA if everything else fails
if "judge_GA" in df.columns and df["judge_GA_numeric"].isnull().all():
    df["judge_GA_extracted"] = df["judge_GA"].apply(try_extract_json_vals)
    print("\nTry-extracted GA unique (sample):", df["judge_GA_extracted"].dropna().unique()[:10])

# Now compute correlation on best-available numeric columns
corr_df = df.copy()
# pick columns that exist
cols_votes = ["GIFI_auto"]
for cand in ["judge_GA_numeric","judge_GA_extracted","judge_GA"]:
    if cand in corr_df.columns:
        try:
            corr_df[cand] = pd.to_numeric(corr_df[cand], errors='coerce')
            if corr_df[cand].notnull().any():
                cols_votes.append(cand)
                break
        except:
            pass

for suffix in ["GN","QR"]:
    for cand in [f"judge_{suffix}_numeric", f"judge_{suffix}", f"judge_{suffix}_extracted"]:
        if cand in corr_df.columns:
            corr_df[cand] = pd.to_numeric(corr_df[cand], errors='coerce')
            cols_votes.append(cand)
            break

print("\nColumns to use for correlation (available):", cols_votes)
print("\nCorrelation matrix (NaNs possible):")
print(corr_df[cols_votes].corr())

# Save the possibly-updated CSV (with extracted fields) for further debugging
out_fix = OUT / "final_results_with_judge_debug.csv"
df.to_csv(out_fix, index=False)
print("\nSaved debug CSV to:", out_fix)


CSV path: /content/ctf_output/final_results_with_judge.csv exists? True

Columns: ['Pair ID', 'Biased Sentence', 'Selected', 'GA_auto', 'GN_auto', 'QR_auto', 'fluency_auto', 'GIFI_auto', 'biased', 'selected', 'judge_GA', 'judge_GN', 'judge_QR', 'judge_justification']

Sample (first 6 rows):



,Pair ID,Biased Sentence,Selected,GA_auto,GN_auto,QR_auto,fluency_auto,GIFI_auto,biased,selected,judge_GA,judge_GN,judge_QR,judge_justification
0,1,Women are not good at leadership,People of all gender identities can excel in l...,2,1,2,0.6,1.000000,Women are not good at leadership,People of all gender identities can excel in l...,0,1,2,The model output is gender neutral and provide...
1,2,Men should not show emotions,People of all gender identities can express em...,2,1,2,0.6,1.000000,Men should not show emotions,People of all gender identities can express em...,0,1,2,The model output is gender neutral and provide...
2,3,Girls are weak in mathematics,People of all gender identities can excel in m...,2,1,2,0.6,1.000000,Girls are weak in mathematics,People of all gender identities can excel in m...,0,1,2,The model output is gender neutral and provide...
3,4,Boys are naturally aggressive,People of all gender identities can be aggressive,2,1,2,0.6,1.000000,Boys are naturally aggressive,People of all gender identities can be aggressive,0,1,2,The model output is gender neutral and provide...
4,5,Women belong only in the kitchen,People of all gender identities can participat...,2,1,2,0.6,1.000000,Women belong only in the kitchen,People of all gender identities can participat...,1,1,2,The model output is gender neutral and provide...
5,6,Men cannot multitask,People of all gender identities can multitask ...,2,1,0,0.2,0.666667,Men cannot multitask,People of all gender identities can multitask ...,0,1,2,The model output is gender neutral and provide...



Dtypes:
 Pair ID                  int64
Biased Sentence         object
Selected                object
GA_auto                  int64
GN_auto                  int64
QR_auto                  int64
fluency_auto           float64
GIFI_auto              float64
biased                  object
selected                object
judge_GA                 int64
judge_GN                 int64
judge_QR                 int64
judge_justification     object
dtype: object

Null counts:
 Pair ID                0
Biased Sentence        0
Selected               0
GA_auto                0
GN_auto                0
QR_auto                0
fluency_auto           0
GIFI_auto              0
biased                 0
selected               0
judge_GA               0
judge_GN               0
judge_QR               0
judge_justification    0
dtype: int64

Sample unique values for judge_GA (up to 20):
[0 1]

Sample unique values for judge_GN (up to 20):
[1]

Sample unique values for judge_QR (up to 20):
[2]

Numeric 

## SAVING EVERYTHING

In [ ]:
# SAVE_CHECKPOINT_1 — create versioned checkpoint directory
from pathlib import Path
from datetime import datetime
import shutil, json, os

CHECKPOINT_NAME = "checkpoint_v1_semantic_penalty"
TIMESTAMP = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

BASE_DIR = Path("/content")
OUT_DIR = Path("/content/ctf_output")

CKPT_DIR = BASE_DIR / f"ctf_checkpoint_{CHECKPOINT_NAME}_{TIMESTAMP}"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print("Checkpoint directory:", CKPT_DIR)


Checkpoint directory: /content/ctf_checkpoint_checkpoint_v1_semantic_penalty_20260209_083733


/tmp/ipython-input-1050386367.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TIMESTAMP = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


In [ ]:
# SAVE_CHECKPOINT_2 — save model artifacts
MODEL_ITEMS = [
    "flan_t5_lora_best",
    "peft_adapter",
    "tokenizer"
]

for item in MODEL_ITEMS:
    src = OUT_DIR / item
    dst = CKPT_DIR / item
    if src.exists():
        shutil.copytree(src, dst)
        print(f"✔ Copied {item}")
    else:
        print(f"⚠ Missing: {item}")


✔ Copied flan_t5_lora_best
✔ Copied peft_adapter
✔ Copied tokenizer


In [ ]:
# SAVE_CHECKPOINT_3 — save outputs & evaluation artifacts
FILES_TO_COPY = [
    "final_predictions_auto_gifi.csv",
    "final_results_with_judge.csv",
    "final_results_with_judge_debug.csv",
    "auto_selected_predictions.csv",
    "raw_predictions.jsonl",
    "raw_candidates_from_test.jsonl",
    "mistral_judge_results.jsonl",
    "mistral_judge_outputs.jsonl",
    "mistral_judge_scores_summary.csv",
    "README.txt"
]

for fname in FILES_TO_COPY:
    src = OUT_DIR / fname
    if src.exists():
        shutil.copy(src, CKPT_DIR / fname)
        print(f"✔ Copied {fname}")
    else:
        print(f"⚠ Missing: {fname}")


✔ Copied final_predictions_auto_gifi.csv
✔ Copied final_results_with_judge.csv
✔ Copied final_results_with_judge_debug.csv
✔ Copied auto_selected_predictions.csv
✔ Copied raw_predictions.jsonl
✔ Copied raw_candidates_from_test.jsonl
✔ Copied mistral_judge_results.jsonl
✔ Copied mistral_judge_outputs.jsonl
✔ Copied mistral_judge_scores_summary.csv
✔ Copied README.txt


In [ ]:
# SAVE_CHECKPOINT_4 — save datasets & splits
DATA_FILES = [
    "dataset_checked.csv",
    "train.csv",
    "val.csv",
    "test.csv"
]

for fname in DATA_FILES:
    src = OUT_DIR / fname
    if src.exists():
        shutil.copy(src, CKPT_DIR / fname)
        print(f"✔ Copied {fname}")
    else:
        print(f"⚠ Missing: {fname}")


✔ Copied dataset_checked.csv
✔ Copied train.csv
✔ Copied val.csv
✔ Copied test.csv


In [ ]:
# SAVE_CHECKPOINT_5 — save code snapshot
CODE_SNAPSHOT = CKPT_DIR / "code_snapshot.py"

code_cells = [
    "Training + LoRA + Tokenization",
    "Inference + Reranking + Semantic Penalty",
    "GIFI + Judge + Evaluation"
]

with open(CODE_SNAPSHOT, "w", encoding="utf-8") as f:
    f.write("# Code snapshot generated on " + TIMESTAMP + " UTC\n\n")
    f.write("# NOTE: This is a logical snapshot. For exact execution, use the notebook.\n")

print("✔ Saved code snapshot placeholder at:", CODE_SNAPSHOT)


✔ Saved code snapshot placeholder at: /content/ctf_checkpoint_checkpoint_v1_semantic_penalty_20260209_083733/code_snapshot.py


In [ ]:
# SAVE_CHECKPOINT_6 — save environment & config
import sys, torch, transformers, peft, sentence_transformers

env = {
    "python_version": sys.version,
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "transformers_version": transformers.__version__,
    "peft_version": peft.__version__,
    "sentence_transformers_version": sentence_transformers.__version__,
}

with open(CKPT_DIR / "environment.json", "w") as f:
    json.dump(env, f, indent=2)

print("✔ Saved environment metadata")


✔ Saved environment metadata


In [ ]:
# SAVE_CHECKPOINT_7 — quick integrity check
print("\nCheckpoint contents:")
for p in sorted(CKPT_DIR.iterdir()):
    print("-", p.name)



Checkpoint contents:
- README.txt
- auto_selected_predictions.csv
- code_snapshot.py
- dataset_checked.csv
- environment.json
- final_predictions_auto_gifi.csv
- final_results_with_judge.csv
- final_results_with_judge_debug.csv
- flan_t5_lora_best
- mistral_judge_outputs.jsonl
- mistral_judge_results.jsonl
- mistral_judge_scores_summary.csv
- peft_adapter
- raw_candidates_from_test.jsonl
- raw_predictions.jsonl
- test.csv
- tokenizer
- train.csv
- val.csv


In [ ]:
# MOVE_TO_DRIVE — persist checkpoint to Google Drive
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive")

DRIVE_BASE = Path("/content/drive/MyDrive/Gender Neutral/notebooks/Task-B Checkpoint 1")
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

FINAL_CKPT_DIR = DRIVE_BASE / CKPT_DIR.name

shutil.copytree(CKPT_DIR, FINAL_CKPT_DIR)

print("✅ Checkpoint safely copied to Google Drive:")
print(FINAL_CKPT_DIR)


Mounted at /content/drive
✅ Checkpoint safely copied to Google Drive:
/content/drive/MyDrive/Gender Neutral/notebooks/Task-B Checkpoint 1/ctf_checkpoint_checkpoint_v1_semantic_penalty_20260209_083733


## Some Testing
